# HoloKai GLiNER Fine-tuning — Mapped Data Only

This notebook treats the **recommended-label mapped datasets as the only formal fine-tuning/evaluation data**.

Formal data:
- `train_gliner_mapping.json`
- `val_gliner_mapping.json`

The original non-mapped GLiNER files are used **only once** to generate/validate the mapped files and are not used in any baseline, smoke test, training, model selection, comparison, or threshold tuning.


## Step 1 — Environment setup

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd "/content/drive/MyDrive/Holokai_Project"


In [ ]:
!pip install -r finetune/requirements-colab.txt


## Step 2 — Generate and validate mapped data

After this step, only the `_mapping.json` files are used by the rest of the notebook.


In [ ]:
!python finetune/scripts/map_recommended_labels.py


In [ ]:
from pathlib import Path
import json
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/Holokai_Project")

ORIG_TRAIN = PROJECT / "finetune/data/train_gliner.json"
ORIG_VAL   = PROJECT / "finetune/data/val_gliner.json"

TRAIN_DATA = PROJECT / "finetune/data/train_gliner_mapping.json"
VAL_DATA   = PROJECT / "finetune/data/val_gliner_mapping.json"

RUN_ROOT = PROJECT / "finetune/outputs/finetune_run_1"
TRAIN_OUT = RUN_ROOT / "training"
BASELINE_OUT = RUN_ROOT / "baseline"
MODEL_EVAL_OUT = RUN_ROOT / "model_eval"
COMPARISON_OUT = RUN_ROOT / "comparison"
THRESHOLD_OUT = RUN_ROOT / "threshold_tuning"

LABEL_MAPPING = {
    "provider": "name medical professional",
    "facility": "organization medical facility",
    "person": "name",
    "location address": "location address",
    "diagnosis": "condition",
    "unique identifier": "personal identifier number",
    "sex": "sex assigned at birth",
    "gender identity": "gender identity",
    "procedure": "medical procedure or test performed",
    "medication": "medication, drug, or prescription",
    "lab result": "lab result or test finding",
}

MAPPED_LABELS = list(LABEL_MAPPING.values())

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def validate_mapping(original_path, mapped_path):
    original = load_json(original_path)
    mapped = load_json(mapped_path)

    assert len(original) == len(mapped)

    for i, (a, b) in enumerate(zip(original, mapped)):
        assert a["tokenized_text"] == b["tokenized_text"], f"Tokens changed at sample {i}"
        assert len(a["ner"]) == len(b["ner"]), f"Entity count changed at sample {i}"

        for old, new in zip(a["ner"], b["ner"]):
            assert old[:2] == new[:2], f"Span changed at sample {i}"
            assert old[2] in LABEL_MAPPING, f"Unexpected original label: {old[2]}"
            assert new[2] == LABEL_MAPPING[old[2]], f"Wrong mapping: {old} -> {new}"

    labels = {e[2] for r in mapped for e in r["ner"]}
    assert labels == set(MAPPED_LABELS), f"Unexpected mapped labels: {labels}"

    print(
        f"PASS: {mapped_path.name} | "
        f"samples={len(mapped)} | "
        f"entities={sum(len(r['ner']) for r in mapped)}"
    )

validate_mapping(ORIG_TRAIN, TRAIN_DATA)
validate_mapping(ORIG_VAL, VAL_DATA)

print("\nFormal train data:", TRAIN_DATA)
print("Formal val data:  ", VAL_DATA)


## Step 3 — Formal training config

The mapped train/validation files are the only data paths used from this point forward.


In [ ]:
BASE_CONFIG = PROJECT / "finetune/configs/train_gliner.json"
RUN_CONFIG = PROJECT / "finetune/configs/finetune_run_1.json"

with open(BASE_CONFIG, encoding="utf-8") as f:
    cfg = json.load(f)

cfg["train_data"] = str(TRAIN_DATA)
cfg["val_data"] = str(VAL_DATA)
cfg["output_dir"] = str(TRAIN_OUT)
cfg["threshold"] = 0.4

# Default formal run hyperparameters
assert cfg["learning_rate"] == 1e-5
assert cfg["batch_size"] == 4
assert cfg["eval_batch_size"] == 8
assert cfg["epochs"] == 3
assert cfg["seed"] == 42

with open(RUN_CONFIG, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("Saved:", RUN_CONFIG)
print(json.dumps(cfg, indent=2))


In [ ]:
!python finetune/scripts/train_gliner.py   --config finetune/configs/finetune_run_1.json   --check-config


## Step 4 — Pretrained baseline on mapped validation data

This is the formal baseline for the fine-tuning experiment.


In [ ]:
!python finetune/scripts/evaluate_gliner.py \
  --label-schema recommended \
  --model knowledgator/gliner-pii-base-v1.0 \
  --data "{VAL_DATA}" \
  --threshold 0.4 \
  --output-dir "{BASELINE_OUT}" \
  --device auto

In [ ]:
import pandas as pd

with open(BASELINE_OUT / "metrics.json", encoding="utf-8") as f:
    baseline_metrics = json.load(f)

print("Pretrained baseline @ threshold=0.4")
print(json.dumps(baseline_metrics["overall"], indent=2))
display(pd.read_csv(BASELINE_OUT / "per_label_metrics.csv"))


## Step 5 — Smoke test using mapped data only

The smoke subsets are sampled directly from the mapped train/validation files.


In [ ]:
SMOKE_ROOT = RUN_ROOT / "smoke_test"
SMOKE_DATA = SMOKE_ROOT / "data"
SMOKE_CONFIG = PROJECT / "finetune/configs/smoke_test_mapping.json"

SMOKE_DATA.mkdir(parents=True, exist_ok=True)

train_full = load_json(TRAIN_DATA)
val_full = load_json(VAL_DATA)

with open(SMOKE_DATA / "train_gliner_mapping.json", "w", encoding="utf-8") as f:
    json.dump(train_full[:100], f, ensure_ascii=False)

with open(SMOKE_DATA / "val_gliner_mapping.json", "w", encoding="utf-8") as f:
    json.dump(val_full[:30], f, ensure_ascii=False)

smoke_cfg = dict(cfg)
smoke_cfg.update({
    "train_data": str(SMOKE_DATA / "train_gliner_mapping.json"),
    "val_data": str(SMOKE_DATA / "val_gliner_mapping.json"),
    "output_dir": str(SMOKE_ROOT / "run"),
    "epochs": 1,
    "eval_steps": 5,
    "logging_steps": 1,
    "threshold": 0.4,
})

with open(SMOKE_CONFIG, "w", encoding="utf-8") as f:
    json.dump(smoke_cfg, f, indent=2)

print("Saved:", SMOKE_CONFIG)


In [ ]:
!python finetune/scripts/train_gliner.py   --config finetune/configs/smoke_test_mapping.json   --check-config


In [ ]:
!python finetune/scripts/train_gliner.py   --config finetune/configs/smoke_test_mapping.json


## Step 6 — Full fine-tuning

Train only on `train_gliner_mapping.json` and validate only on `val_gliner_mapping.json`.


In [ ]:
!python finetune/scripts/train_gliner.py   --config finetune/configs/finetune_run_1.json


In [ ]:
!find "{TRAIN_OUT}" -maxdepth 2 -type d -print
!ls -lh "{TRAIN_OUT}"


In [ ]:
history = load_json(TRAIN_OUT / "training_history.json")

provider_label = LABEL_MAPPING["provider"]

for row in history:
    if "eval_f1" not in row:
        continue

    provider_recall = None
    per_label = row.get("eval_exact_metrics", {}).get("per_label", {})
    if provider_label in per_label:
        provider_recall = per_label[provider_label].get("recall")

    print(
        f"step={row.get('step')} | "
        f"epoch={row.get('epoch')} | "
        f"F1={row['eval_f1']:.4f} | "
        f"P={row['eval_precision']:.4f} | "
        f"R={row['eval_recall']:.4f} | "
        f"PROVIDER_R={provider_recall}"
    )


## Step 7 — Select the best saved model

Evaluate every saved epoch model and `final_model` on the same mapped validation data at threshold `0.4`.


In [ ]:
import sys
sys.path.insert(0, str(PROJECT / "finetune/scripts"))

from label_schemas import SCHEMAS, get_labels

MAPPED_SCHEMA = next(
    schema
    for schema in SCHEMAS
    if "medical procedure or test performed" in get_labels(schema)
)

print("Using label schema:", MAPPED_SCHEMA)

MODEL_EVAL_OUT.mkdir(parents=True, exist_ok=True)

candidates = sorted(
    [
        p.name for p in TRAIN_OUT.iterdir()
        if p.is_dir() and (p.name.startswith("epoch-") or p.name == "final_model")
    ],
    key=lambda x: (x == "final_model", x)
)

print("Candidates:", candidates)

for model_name in candidates:
    model_path = TRAIN_OUT / model_name
    output_path = MODEL_EVAL_OUT / model_name

    !python finetune/scripts/evaluate_gliner.py \
        --label-schema "{MAPPED_SCHEMA}" \
        --model "{model_path}" \
        --data "{VAL_DATA}" \
        --threshold 0.4 \
        --output-dir "{output_path}" \
        --device auto

In [ ]:
rows = []

for model_name in candidates:
    d = MODEL_EVAL_OUT / model_name

    with open(d / "metrics.json", encoding="utf-8") as f:
        metrics = json.load(f)

    label_df = pd.read_csv(d / "per_label_metrics.csv")
    provider = label_df.loc[
        label_df["label"].str.lower() == provider_label.lower()
    ].iloc[0]

    rows.append({
        "model": model_name,
        "precision": metrics["overall"]["precision"],
        "recall": metrics["overall"]["recall"],
        "f1": metrics["overall"]["f1"],
        "provider_precision": float(provider["precision"]),
        "provider_recall": float(provider["recall"]),
        "provider_f1": float(provider["f1"]),
    })

model_table = pd.DataFrame(rows).sort_values(
    ["f1", "provider_f1", "precision"],
    ascending=False
).reset_index(drop=True)

display(model_table)

BEST_MODEL_NAME = model_table.loc[0, "model"]
BEST_MODEL_PATH = TRAIN_OUT / BEST_MODEL_NAME

print("Selected best model:", BEST_MODEL_NAME)
print("Path:", BEST_MODEL_PATH)

with open(RUN_ROOT / "best_model.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": BEST_MODEL_NAME,
        "path": str(BEST_MODEL_PATH),
        "selection": "highest validation F1 @ 0.4; ties by provider F1 then precision",
    }, f, indent=2)


## Step 8 — Pretrained vs fine-tuned comparison

Both sides use:
- mapped labels,
- mapped validation data,
- threshold `0.4`,
- identical exact-span evaluation.


In [ ]:
COMPARISON_OUT.mkdir(parents=True, exist_ok=True)

with open(BASELINE_OUT / "metrics.json", encoding="utf-8") as f:
    base_overall = json.load(f)["overall"]

with open(MODEL_EVAL_OUT / BEST_MODEL_NAME / "metrics.json", encoding="utf-8") as f:
    ft_overall = json.load(f)["overall"]

overall_comparison = {
    "threshold": 0.4,
    "pretrained": base_overall,
    "fine_tuned": ft_overall,
    "delta": {
        "precision": ft_overall["precision"] - base_overall["precision"],
        "recall": ft_overall["recall"] - base_overall["recall"],
        "f1": ft_overall["f1"] - base_overall["f1"],
    },
}

with open(COMPARISON_OUT / "pretrained_vs_finetuned_overall.json", "w", encoding="utf-8") as f:
    json.dump(overall_comparison, f, indent=2)

base_df = pd.read_csv(BASELINE_OUT / "per_label_metrics.csv")
ft_df = pd.read_csv(MODEL_EVAL_OUT / BEST_MODEL_NAME / "per_label_metrics.csv")

comparison_df = base_df.merge(
    ft_df,
    on="label",
    suffixes=("_pretrained", "_finetuned"),
)

for metric in ["precision", "recall", "f1"]:
    comparison_df[f"{metric}_delta"] = (
        comparison_df[f"{metric}_finetuned"]
        - comparison_df[f"{metric}_pretrained"]
    )

comparison_df.to_csv(
    COMPARISON_OUT / "pretrained_vs_finetuned_per_label.csv",
    index=False
)

print(json.dumps(overall_comparison, indent=2))
display(comparison_df)


## Step 9 — Per-label threshold tuning

Tune thresholds only for the selected best fine-tuned model, using the mapped validation data.


In [ ]:
!python finetune/scripts/tune_per_label_thresholds.py \
  --label-schema "{MAPPED_SCHEMA}" \
  --model "{BEST_MODEL_PATH}" \
  --data "{VAL_DATA}" \
  --output-dir "{THRESHOLD_OUT}"

In [ ]:
best_thresholds = load_json(THRESHOLD_OUT / "best_thresholds.json")
calibrated_metrics = load_json(THRESHOLD_OUT / "calibrated_metrics.json")

print("Best thresholds:")
print(json.dumps(best_thresholds, indent=2))

print("\nCalibrated validation overall:")
print(json.dumps(calibrated_metrics["overall"], indent=2))

display(pd.read_csv(THRESHOLD_OUT / "calibrated_per_label_metrics.csv"))


## Step 10 — Save the run summary

The calibrated score is validation-calibrated performance, not independent held-out test performance.


In [ ]:
summary = {
    "run": "finetune_run_1",
    "data_policy": "mapped data only",
    "train_data": str(TRAIN_DATA),
    "val_data": str(VAL_DATA),
    "base_model": cfg["base_model"],
    "labels": MAPPED_LABELS,
    "train_samples": len(load_json(TRAIN_DATA)),
    "val_samples": len(load_json(VAL_DATA)),
    "hyperparameters": {
        "learning_rate": cfg["learning_rate"],
        "batch_size": cfg["batch_size"],
        "eval_batch_size": cfg["eval_batch_size"],
        "epochs": cfg["epochs"],
        "seed": cfg["seed"],
    },
    "selected_model": BEST_MODEL_NAME,
    "fixed_threshold": 0.4,
    "pretrained_overall_at_0_4": base_overall,
    "finetuned_overall_at_0_4": ft_overall,
    "finetuning_delta_at_0_4": overall_comparison["delta"],
    "best_thresholds": best_thresholds.get("thresholds", best_thresholds),
    "calibrated_validation_overall": calibrated_metrics["overall"],
    "note": "All formal training/evaluation uses mapped labels only. Thresholds are tuned on validation data.",
}

with open(RUN_ROOT / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved:", RUN_ROOT / "run_summary.json")
